# 02 Customer base

**Project question.** With a limited marketing budget, which customers should receive an SMS so that it creates the most additional purchases?

**This notebook.** Describes who the customers are, using only information known before the SMS. The features built here feed every later model, and the segments describe the audience a targeting policy selects.

**Summary.** All 400,162 clients in the data shopped in the three weeks before the SMS, so the data covers active clients only. Spend is only moderately concentrated: the top 20% of clients make about half of it. Seven stable behaviour segments emerge, two of which (point redeemers and express point collectors) may be the clients who react to promotions.

**Research question.** How do customers differ in their shopping behaviour, and how concentrated is their value?

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from x5uplift import plotting as P
from x5uplift.config import CATEGORY_MAP_PATH, FEATURE_GROUPS_PATH, FEATURES_PATH, SEGMENTS_PATH
from x5uplift.segments import (FEATURE_LABELS, SEGMENT_FEATURES, SEGMENT_NAMES,
                               choose_k, fit_segments, prepare_matrix, profile)

pd.set_option("display.width", 140)
P.set_style()

features = pd.read_parquet(FEATURES_PATH)  # built by scripts/04_build_features.py
features.shape

## 1. Client features

`features.py` builds 52 features per client from data up to the decision moment, in eight groups. Two cleaning rules from notebook 01 are applied there: redemption dates after the decision moment count as "not redeemed yet", and ages outside 14 to 100 are missing. 536 clients redeem points a few seconds before their card issue time (same checkout), so that gap is clipped at zero.

In [ ]:
groups = json.loads(FEATURE_GROUPS_PATH.read_text())
share_cols = [c for c in features if c.startswith("share_") or c == "top_store_share"]
assert features["client_id"].is_unique and len(features) == 400_162
assert all(features[c].dropna().between(0, 1).all() for c in share_cols)
pd.Series({g: len(c) for g, c in groups.items()}, name="n_features")

Category features use the 12 largest `level_2` categories by spend plus an "other" share. Category ids are hashed, so features are named by spend rank:

In [ ]:
cats = pd.read_csv(CATEGORY_MAP_PATH)
cats["cum_spend_share"] = cats["spend_share"].cumsum()
cats.loc[cats["cat_rank"].isin([1, 4, 8, 12, 20, 42]), ["cat_rank", "spend_share", "cum_spend_share"]].round(3)

The top 12 categories cover 91% of spend; the remaining 30 are grouped as "other".

## 2. Who the clients are

In [ ]:
gender_share = features["gender"].value_counts(normalize=True).reindex(["F", "M", "U"]) * 100
redeem_share = features["redeemed_before_cutoff"].mean() * 100
card_years = features["days_since_card_issue"] / 365.25

fig, ((ax_age, ax_gender), (ax_card, ax_points)) = plt.subplots(2, 2, figsize=(10, 6.5))

ax_age.hist(features["age"].dropna(), bins=np.arange(14, 102, 1), color=P.BLUE, edgecolor=P.SURFACE, linewidth=0.5)
ax_age.set(xlabel="Age (years)", ylabel="Clients")
ax_age.set_title(f"Median age {features['age'].median():.0f}, with two peaks", loc="left", fontsize=11)

names = ["Female", "Male", "Not recorded"]
ax_gender.barh(names, gender_share.values, color=[P.BLUE, P.BLUE, P.MUTED], height=0.5)
ax_gender.set_title(f"Gender is not recorded for {gender_share['U']:.0f}%", loc="left", fontsize=11)

ax_card.hist(card_years, bins=np.arange(0, 2.01, 1 / 12), color=P.BLUE, edgecolor=P.SURFACE, linewidth=0.5)
ax_card.set(xlabel="Card age at the decision moment (years)", ylabel="Clients")
ax_card.set_title(f"{(card_years > 1).mean() * 100:.0f}% of cards are over a year old, most from the 2017 launch",
                  loc="left", fontsize=11)

ax_points.barh(["Has redeemed points", "Never redeemed"], [redeem_share, 100 - redeem_share], color=P.BLUE, height=0.5)
ax_points.set_title(f"{redeem_share:.0f}% had used points before the decision moment", loc="left", fontsize=11)

for ax, values in ((ax_gender, gender_share.values), (ax_points, [redeem_share, 100 - redeem_share])):
    for y, v in enumerate(values):
        ax.text(v + 1, y, f"{v:.0f}%", va="center", fontsize=10)
    ax.invert_yaxis()
    ax.set(xlim=(0, 100), xlabel="Share of clients (%)")
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", visible=True)
for ax in (ax_age, ax_card):
    ax.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}K" if v >= 1000 else f"{v:.0f}")

fig.suptitle("The typical client is middle aged, has held the card for over a year and already uses points",
             x=0.01, y=0.995, ha="left", va="top", fontsize=13, fontweight="bold")
fig.text(0.01, 0.95, f"All {len(features):,} clients, information known at the decision moment",
         ha="left", va="top", fontsize=10, color=P.INK_2)
fig.tight_layout(rect=(0, 0, 1, 0.91))
P.save(fig, "02_client_profile")
plt.show()

## 3. How clients shop

In [ ]:
rfm = features[["days_since_last_purchase", "n_baskets", "total_spend"]]
med = rfm.median()
rec = rfm["days_since_last_purchase"]
cap = int(rfm["n_baskets"].quantile(0.99))
spend = rfm["total_spend"][rfm["total_spend"] > 0]
lo, hi = spend.quantile([0.001, 0.999])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].hist(rec, bins=np.arange(0, np.ceil(rec.max()) + 1, 1), color=P.BLUE, edgecolor=P.SURFACE, linewidth=0.5)
axes[0].axvline(rec.max(), color=P.INK_2, linewidth=1)
axes[0].text(rec.max(), axes[0].get_ylim()[1] * 0.97, f"Longest gap\n{rec.max():.1f} days ",
             ha="right", va="top", fontsize=9, color=P.INK_2)
axes[0].set(xlim=(0, np.ceil(rec.max()) + 3), xlabel="Days since last purchase", ylabel="Clients")
axes[0].set_title(f"Recency: median {med['days_since_last_purchase']:.0f} days", loc="left", fontsize=11)

axes[1].hist(rfm["n_baskets"].clip(upper=cap), bins=np.arange(0.5, cap + 1.5, 1), color=P.BLUE)
axes[1].set_xlabel(f"Baskets in 4 months (last bar: {cap} or more)")
axes[1].set_title(f"Frequency: median {med['n_baskets']:.0f} baskets", loc="left", fontsize=11)

axes[2].hist(spend.clip(lo, hi), bins=np.logspace(np.log10(lo), np.log10(hi), 50), color=P.BLUE,
             edgecolor=P.SURFACE, linewidth=0.5)
axes[2].set_xscale("log")
axes[2].xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[2].set_xlabel("Total spend in 4 months (log scale, 0.1% tails clipped)")
axes[2].set_title(f"Spend: median {med['total_spend']:,.0f}", loc="left", fontsize=11)

for ax in axes:
    ax.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}K" if v >= 1000 else f"{v:.0f}")
fig.suptitle(f"Every client shopped in the last {rec.max():.0f} days: the data covers active clients only",
             x=0.01, y=0.995, ha="left", va="top", fontsize=13, fontweight="bold")
fig.text(0.01, 0.935, f"Recency, frequency and spend (RFM) of all {len(rfm):,} clients at the decision moment",
         ha="left", va="top", fontsize=10, color=P.INK_2)
fig.tight_layout(rect=(0, 0, 1, 0.89))
P.save(fig, "02_rfm")
plt.show()

In [ ]:
features[["days_since_last_purchase", "days_since_first_purchase"]].describe(percentiles=[0.5, 0.99, 0.999]).T.round(1)

**The data covers active clients only.** No client is more than 22.1 days past their last purchase, while the history starts on the same date for everyone. X5 most likely selected clients who shopped in roughly the last three weeks (the rule is not documented). Results therefore describe recently active clients, churn cannot be studied, and the high purchase rate after the SMS (62%) partly reflects this selection.

## 4. How concentrated is value?

In [ ]:
spend_sorted = np.sort(features["total_spend"].to_numpy())
n = len(spend_sorted)
cum_clients = np.concatenate([[0], np.arange(1, n + 1) / n])
cum_spend = np.concatenate([[0], np.cumsum(spend_sorted) / spend_sorted.sum()])
gini = 1 - 2 * np.sum((cum_spend[1:] + cum_spend[:-1]) / 2 * np.diff(cum_clients))
top20, top10 = (1 - cum_spend[int(round((1 - p) * n))] for p in (0.20, 0.10))

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 100], [0, 100], color=P.AXIS, linewidth=1)
ax.plot(cum_clients * 100, cum_spend * 100, color=P.BLUE)
ax.scatter([80], [(1 - top20) * 100], s=40, color=P.BLUE, edgecolor=P.SURFACE, linewidth=1.5, zorder=3)
ax.annotate(f"Top 20% of clients: {top20 * 100:.0f}% of spend\nTop 10% of clients: {top10 * 100:.0f}% of spend",
            xy=(80, (1 - top20) * 100), xytext=(98, 5), ha="right", fontsize=9, color=P.INK_2,
            arrowprops=dict(arrowstyle="-", color=P.MUTED, linewidth=1))
ax.text(50, 53, "Equal spend for everyone", rotation=45, fontsize=8, color=P.INK_2, ha="center")
ax.set(xlim=(0, 100), ylim=(0, 100), xlabel="Clients, sorted from lowest to highest spend (cumulative %)",
       ylabel="Share of total spend (cumulative %)")
ax.grid(axis="x", visible=True)
P.titles(ax, f"The top 20% of clients make {top20 * 100:.0f}% of all spend",
         f"Spend in the 4 months before the decision, all {n:,} clients; Gini {gini:.2f}")
fig.tight_layout()
P.save(fig, "02_value_concentration")
plt.show()

Value is only moderately concentrated (Gini 0.47), as expected for a grocer where most clients shop regularly. The 4 month window understates the value of occasional shoppers, while the selection of active clients leaves out low value lapsed clients; the two effects pull in opposite directions. Value alone does not single out a small group worth contacting.

## 5. Segments

k-means on 10 standardized behaviour features (`segments.py`). For each k, separation (silhouette) and stability (adjusted Rand index between runs on different subsamples and seeds, 1 = identical segments) are measured.

In [ ]:
X_seg = prepare_matrix(features)
choose_k(X_seg).round(3)

k = 3, 7 and 8 are stable; k = 4 to 6 are not. k = 5 has the best separation but gives different segments on different subsamples, so separation alone would mislead. k = 7 is chosen: as stable as k = 8, and richer than k = 3.

In [ ]:
labels = fit_segments(X_seg, 7)
prof = profile(features, labels)

# The names in segments.py were given by reading this profile; stop if a refit reorders the segments
assert prof["days_since_last_purchase"].idxmin() == 0
assert prof["avg_basket_spend"].idxmax() == 2
assert prof["share_alcohol"].idxmax() == 4
assert prof["share_baskets_points_spent"].idxmax() == 5
assert prof["share_baskets_express_received"].idxmax() == 6

segments = pd.DataFrame({"client_id": features["client_id"], "segment": labels})
segments["segment_name"] = segments["segment"].map(SEGMENT_NAMES)
segments.to_parquet(SEGMENTS_PATH, index=False)
prof.rename(index=SEGMENT_NAMES).T.round(2)

In [ ]:
z = pd.DataFrame(X_seg, columns=SEGMENT_FEATURES).groupby(labels).mean()  # mean in SD units
sizes = pd.Series(labels).value_counts(normalize=True).sort_index()
cmap = LinearSegmentedColormap.from_list("blue_grey_red", ["#184f95", "#86b6ef", "#f0efec", "#f09a99", "#b8302f"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(z.clip(-2, 2).to_numpy(), cmap=cmap, vmin=-2, vmax=2, aspect="auto")
for (i, j), v in np.ndenumerate(z.to_numpy()):
    ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=9, color="white" if abs(v) > 1.2 else P.INK)
ax.set_xticks(range(z.shape[1]), [FEATURE_LABELS[c] for c in z.columns], rotation=30, ha="right")
ax.set_yticks(range(z.shape[0]), [f"{SEGMENT_NAMES[s]} ({sizes[s] * 100:.0f}%)" for s in z.index])
ax.grid(False)
ax.spines[["left", "bottom"]].set_visible(False)
ax.tick_params(length=0)
P.titles(ax, "Seven stable segments, from daily top-up shoppers to express point collectors",
         "Segment average minus the average client, in standard deviations (red above, blue below; colours capped at ±2)")
fig.tight_layout()
P.save(fig, "02_segments")
plt.show()

Three segments differ in how often and how much clients buy (daily top-up, big basket, occasional), one is close to the average client, and three small segments are each defined by a single behaviour: alcohol, redeeming points, collecting express points. Weekend shopping separates no segment.

In [ ]:
seg_value = (features[["total_spend"]].assign(segment=segments["segment_name"].values)
             .groupby("segment")["total_spend"].agg(["size", "sum", "median"]))
seg_value = pd.DataFrame({
    "share_of_clients": seg_value["size"] / seg_value["size"].sum(),
    "share_of_spend": seg_value["sum"] / seg_value["sum"].sum(),
    "median_spend": seg_value["median"],
}).sort_values("share_of_spend", ascending=False)
seg_value.round(3)

Big basket and daily top-up shoppers are 47% of clients but make 74% of spend. Occasional shoppers are 16% of clients and 4% of spend.

## Conclusion

**Answer.** The clients are recently active loyalty card holders whose value is only moderately concentrated (top 20% make 51% of spend). They fall into seven stable segments: three by frequency and basket size, one typical group, and three small groups defined by alcohol buying, point redemption and express points.

**What we learned**
* The data contains only clients active in the last three weeks, which limits what the results can say about the full customer base.
* Stability checks matter: the best separated k was not reproducible.
* Two segments are linked to promotions and are natural candidates for a larger SMS effect.

**What did not work or stays open**
* Churn cannot be studied with this selection.
* Category ids are hashed, so category features cannot be named.

**Next step.** Notebook 03 predicts who buys without an SMS. Notebook 04 then tests whether the SMS effect differs across these segments.